In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import zarr
import tqdm
from torch.utils.data import Dataset, DataLoader

In [7]:
# Ersetzen Sie die alte, kompakte UNet3D-Klasse durch diese klare und korrigierte Version.
class UNet3D(nn.Module):
    """
    Eine klare und korrekte Definition des 3D U-Nets.
    """
    def __init__(self, in_channels=2, out_channels=3):
        super().__init__()
        
        # Encoder Path
        self.enc1 = self._conv_block(in_channels, 16) # Output: 16 Kanäle
        self.pool1 = nn.MaxPool3d(2)
        self.enc2 = self._conv_block(16, 32) # Output: 32 Kanäle
        self.pool2 = nn.MaxPool3d(2)
        
        # Bottleneck
        self.bottleneck = self._conv_block(32, 64)
        
        # Decoder Path
        self.upconv2 = nn.ConvTranspose3d(64, 32, kernel_size=2, stride=2)
        # KORREKTUR HIER: Nach der Verkettung haben wir 32 (upconv) + 32 (skip) = 64 Kanäle
        self.dec2 = self._conv_block(64, 32) 
        
        self.upconv1 = nn.ConvTranspose3d(32, 16, kernel_size=2, stride=2)
        # KORREKTUR HIER: Nach der Verkettung haben wir 16 (upconv) + 16 (skip) = 32 Kanäle
        self.dec1 = self._conv_block(32, 16)
        
        self.final_conv = nn.Conv3d(16, out_channels, kernel_size=1)
        self.final_conv.weight.data.zero_()
        self.final_conv.bias.data.zero_()

    def _conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv3d(in_c, out_c, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm3d(out_c), # InstanceNorm ist oft besser für Registrierung als BatchNorm
            nn.ReLU(inplace=True),
            nn.Conv3d(out_c, out_c, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm3d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x_fixed, x_moving):
        # Verkette die Inputs
        x = torch.cat([x_fixed, x_moving], dim=1)
        
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        
        # Bottleneck
        b = self.bottleneck(self.pool2(e2))
        
        # Decoder mit Skip Connections
        d2 = self.upconv2(b)
        # Prüfen Sie die Größen vor der Verkettung
        if d2.shape[2:] != e2.shape[2:]:
             # Passe die Größe von e2 an d2 an, falls Padding-Fehler auftreten
             e2 = F.interpolate(e2, size=d2.shape[2:], mode='trilinear', align_corners=False)
        d2_cat = torch.cat([d2, e2], dim=1)
        d2_out = self.dec2(d2_cat)
        
        d1 = self.upconv1(d2_out)
        if d1.shape[2:] != e1.shape[2:]:
            e1 = F.interpolate(e1, size=d1.shape[2:], mode='trilinear', align_corners=False)
        d1_cat = torch.cat([d1, e1], dim=1)
        d1_out = self.dec1(d1_cat)
        
        return self.final_conv(d1_out)
    
class SpatialTransformer3D(nn.Module):
    def __init__(self, size):
        super().__init__()
        vectors = [torch.arange(0, s) for s in size]
        grids = torch.meshgrid(vectors, indexing='ij')
        grid = torch.stack(grids)
        grid = grid.unsqueeze(0)
        self.register_buffer('grid', grid.float(), persistent=False)

    def forward(self, src, flow):
        new_locs = self.grid + flow
        shape = flow.shape[2:]
        for i in range(len(shape)):
            new_locs[:, i, ...] = 2 * (new_locs[:, i, ...] / (shape[i] - 1) - 0.5)
        new_locs = new_locs.permute(0, 2, 3, 4, 1)
        new_locs = new_locs[..., [2, 1, 0]]
        return F.grid_sample(src, new_locs, align_corners=True, padding_mode="border")

In [8]:
class LocalNCCLoss(nn.Module):
    def __init__(self, window_size=9):
        super().__init__()
        self.window_size = window_size
        self.padding = window_size // 2
        self.avg_pool = nn.AvgPool3d(self.window_size, 1, self.padding)
    def forward(self, i1, i2):
        mu1 = self.avg_pool(i1)
        mu2 = self.avg_pool(i2)
        s1_sq = self.avg_pool(i1*i1) - mu1*mu1
        s2_sq = self.avg_pool(i2*i2) - mu2*mu2
        cov = self.avg_pool(i1*i2) - mu1*mu2
        num = cov*cov
        den = s1_sq*s2_sq
        return 1 - torch.mean(num / (den + 1e-6))

In [9]:
def smooth_loss(displacement_field):
    dy = torch.abs(displacement_field[:, :, 1:, :, :] - displacement_field[:, :, :-1, :, :])
    dx = torch.abs(displacement_field[:, :, :, 1:, :] - displacement_field[:, :, :, :, 1:])
    dz = torch.abs(displacement_field[:, :, :, :, 1:] - displacement_field[:, :, :, :, :-1]) # Fehler hier korrigiert
    return (torch.mean(dx**2) + torch.mean(dy**2) + torch.mean(dz**2)) / 3.0

In [10]:
class PairwiseDataset(Dataset):
    def __init__(self, moving_zarr_path, fixed_zarr_path, fixed_index=0):
        self.moving_arr = zarr.open(moving_zarr_path, mode='r')
        self.fixed_arr = zarr.open(fixed_zarr_path, mode='r')
        self.num_images = self.moving_arr.shape[3]
        
        self.original_shape = self.moving_arr.shape
        self.padded_shape = [s for s in self.original_shape[:3]]
        for i in range(3):
            if self.padded_shape[i] % 4 != 0:
                self.padded_shape[i] = (self.padded_shape[i] // 4 + 1) * 4
        
        self.fixed_np = self.fixed_arr[..., fixed_index]
        self.fixed_tensor = self._preprocess(self.fixed_np)

    def __len__(self):
        return self.num_images

    def __getitem__(self, idx):
        moving_np = self.moving_arr[..., idx]
        moving_tensor = self._preprocess(moving_np)
        return moving_tensor, self.fixed_tensor

    def _preprocess(self, volume_np):
        # Hier bei Bedarf Normalisierung einfügen
        tensor = torch.from_numpy(volume_np.astype(np.float32)).permute(2, 0, 1).unsqueeze(0)
        pad_d = self.padded_shape[2] - tensor.shape[1]
        pad_h = self.padded_shape[1] - tensor.shape[2] # H ist jetzt an Index 1
        pad_w = self.padded_shape[0] - tensor.shape[3] # W ist jetzt an Index 0
        padding = (pad_w // 2, pad_w - pad_w // 2, pad_h // 2, pad_h - pad_h // 2, pad_d // 2, pad_d - pad_d // 2)
        return F.pad(tensor, padding, "constant", 0)

In [11]:
# =================================================================================
# 2. NEUES GROUPWISE-MODELL
# =================================================================================
class GroupwiseRegistrationModel(nn.Module):
    def __init__(self, input_size, mini_batch_size=16): # Neuer Parameter für die Batch-Größe
        super().__init__()
        self.registration_net = UNet3D()
        self.spatial_transformer = SpatialTransformer3D(size=input_size)
        self.mini_batch_size = mini_batch_size

    def forward(self, group_of_images):
        num_in_group = group_of_images.shape[0]
        
        # 1. Erzeuge das Template aus der gesamten Gruppe (passt noch in den VRAM)
        template = torch.mean(group_of_images, dim=0, keepdim=True)
        
        # Initialisiere Listen, um die Ergebnisse der Mini-Batches zu sammeln
        all_warped_images = []
        all_predicted_dvfs = []
        
        # 2. Schleife über die Gruppe in kleineren Mini-Batches
        for i in range(0, num_in_group, self.mini_batch_size):
            # Wähle einen kleinen Teil der Bilder aus
            sub_batch_images = group_of_images[i : i + self.mini_batch_size]
            current_sub_batch_size = sub_batch_images.shape[0]

            # Erzeuge nur so viele Template-Kopien, wie für diesen Mini-Batch nötig
            sub_template_batch = template.repeat(current_sub_batch_size, 1, 1, 1, 1)

            # Sage die DVFs für den Mini-Batch voraus
            sub_dvfs = self.registration_net(sub_template_batch, sub_batch_images)
            all_predicted_dvfs.append(sub_dvfs)

            # Wende die DVFs an
            sub_warped = self.spatial_transformer(sub_batch_images, sub_dvfs)
            all_warped_images.append(sub_warped)
            
        # Füge die Ergebnisse der Mini-Batches zu einem großen Tensor zusammen
        final_warped_images = torch.cat(all_warped_images, dim=0)
        final_predicted_dvfs = torch.cat(all_predicted_dvfs, dim=0)
        
        # Erzeuge das finale Template-Batch für die Loss-Berechnung
        final_template_batch = template.repeat(num_in_group, 1, 1, 1, 1)

        return final_warped_images, final_template_batch, final_predicted_dvfs


In [12]:
if __name__ == '__main__':
    # --- Konfiguration ---
    MOVING_ZARR_PATH = "/media/shooty/Dev/repos/MRI-MoCoCo/MRI-Datasets/DCE"
    FIXED_ZARR_PATH = "/media/shooty/Dev/repos/MRI-MoCoCo/MRI-Datasets/mdreg_DCE_fitting_results/coreg_zarr_2.zarr"
    MODEL_SAVE_PATH = "./pairwise_model_final.pth"
    FIXED_IMAGE_INDEX = 50
    LEARNING_RATE = 1e-4
    NUM_EPOCHS = 50
    SMOOTH_REG_WEIGHT = 1e-3
    BATCH_SIZE = 2

    # --- Setup ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Verwende Gerät: {device}")

    dataset = PairwiseDataset(MOVING_ZARR_PATH, FIXED_ZARR_PATH, fixed_index=FIXED_IMAGE_INDEX)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

    # --- Modell initialisieren ---
    padded_shape_dims = dataset.padded_shape # H, W, D
    # input_size für STN ist (D, H, W)
    padded_input_size = (padded_shape_dims[2], padded_shape_dims[0], padded_shape_dims[1])
    
    model = UNet3D().to(device)
    stn = SpatialTransformer3D(size=padded_input_size).to(device)
    try:
        model = torch.compile(model)
        print("Modell optimiert.")
    except Exception:
        print("torch.compile() nicht verfügbar.")
    
    similarity_loss_fn = LocalNCCLoss().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # --- Trainings-Schleife ---
    print(f"Starte Pairwise Training mit Batch-Größe: {BATCH_SIZE}")
    for epoch in range(NUM_EPOCHS):
        pbar = tqdm.tqdm(dataloader, desc=f"Epoche {epoch+1}/{NUM_EPOCHS}")
        for moving_batch, fixed_batch in pbar:
            # Squeeze, um die überflüssige Channel-Dimension zu entfernen, die vom unsqueeze(0) im Dataset kommt
            moving_batch = moving_batch.squeeze(1).to(device)
            fixed_batch = fixed_batch.squeeze(1).to(device)

            optimizer.zero_grad()
            
            # Forward-Pass
            predicted_dvfs = model(fixed_batch, moving_batch)
            warped_images = stn(moving_batch, predicted_dvfs)
            
            # Verlustberechnung
            sim_loss = similarity_loss_fn(warped_images, fixed_batch)
            sm_loss = smooth_loss(predicted_dvfs)
            total_loss = sim_loss + SMOOTH_REG_WEIGHT * sm_loss
            
            total_loss.backward()
            optimizer.step()
            
            pbar.set_postfix(loss=total_loss.item())

    torch.save(model.state_dict(), MODEL_SAVE_PATH)
    print(f"✔️ Training abgeschlossen.")

Verwende Gerät: cuda
Modell optimiert.
Starte Pairwise Training mit Batch-Größe: 2


Epoche 1/50:   0%|          | 0/500 [00:00<?, ?it/s]


TorchRuntimeError: Dynamo failed to run FX node with fake tensors: call_function <built-in method conv3d of type object at 0x7efea6ef5fc0>(*(FakeTensor(..., device='cuda:0', size=(32, 104, 128, 128),
           grad_fn=<CatBackward0>), Parameter(FakeTensor(..., device='cuda:0', size=(32, 64, 3, 3, 3), requires_grad=True)), None, (1, 1, 1), (1, 1, 1), (1, 1, 1), 1), **{}): got RuntimeError('Given groups=1, weight of size [32, 64, 3, 3, 3], expected input[1, 32, 104, 128, 128] to have 64 channels, but got 32 channels instead')

from user code:
   File "/tmp/ipykernel_4683/2548246855.py", line 59, in forward
    d2_out = self.dec2(d2_cat)
  File "/home/shooty/anaconda3/envs/ds-env-01/lib/python3.13/site-packages/torch/nn/modules/container.py", line 240, in forward
    input = module(input)
  File "/home/shooty/anaconda3/envs/ds-env-01/lib/python3.13/site-packages/torch/nn/modules/conv.py", line 725, in forward
    return self._conv_forward(input, self.weight, self.bias)
  File "/home/shoo

In [2]:
class UNet3D(nn.Module):
    def __init__(self, in_channels=2, out_channels=3):
        super().__init__();self.enc1=self._conv_block(in_channels,16);self.enc2=self._conv_block(16,32);self.pool1=nn.MaxPool3d(2);self.pool2=nn.MaxPool3d(2);self.bottleneck=self._conv_block(32,64);self.upconv2=nn.ConvTranspose3d(64,32,kernel_size=2,stride=2);self.dec2=self._conv_block(64,32);self.upconv1=nn.ConvTranspose3d(32,16,kernel_size=2,stride=2);self.dec1=self._conv_block(32,16);self.final_conv=nn.Conv3d(16,out_channels,kernel_size=1);self.final_conv.weight.data.zero_();self.final_conv.bias.data.zero_()
    def _conv_block(self,in_c,out_c):return nn.Sequential(nn.Conv3d(in_c,out_c,3,1,1,bias=False),nn.InstanceNorm3d(out_c),nn.ReLU(True),nn.Conv3d(out_c,out_c,3,1,1,bias=False),nn.InstanceNorm3d(out_c),nn.ReLU(True))
    def forward(self,x_fixed,x_moving):
        x=torch.cat([x_fixed,x_moving],dim=1);e1=self.enc1(x);e2=self.enc2(self.pool1(e1));b=self.bottleneck(self.pool2(e2));d2=self.upconv2(b)
        if d2.shape[2:]!=e2.shape[2:]:d2=F.interpolate(d2,size=e2.shape[2:],mode='trilinear',align_corners=False)
        d2_cat=torch.cat([d2,e2],dim=1);d2_out=self.dec2(d2_cat);d1=self.upconv1(d2_out)
        if d1.shape[2:]!=e1.shape[2:]:d1=F.interpolate(d1,size=e1.shape[2:],mode='trilinear',align_corners=False)
        d1_cat=torch.cat([d1,e1],dim=1);d1_out=self.dec1(d1_cat);return self.final_conv(d1_out)

class SpatialTransformer3D(nn.Module):
    def __init__(self,size):
        super().__init__();v=[torch.arange(0,s)for s in size];g=torch.meshgrid(v,indexing='ij');g=torch.stack(g).unsqueeze(0);self.register_buffer('grid',g.float(),persistent=False)
    def forward(self,s,f):
        n=self.grid+f;sh=f.shape[2:];[setattr(n[:,i,...],'data',2*(n[:,i,...]/(sh[i]-1)-0.5))for i in range(len(sh))];n=n.permute(0,2,3,4,1);n=n[...,[2,1,0]];return F.grid_sample(s,n,align_corners=True,padding_mode="border")

class LocalNCCLoss(nn.Module):
    def __init__(self,window_size=9):super().__init__();self.window_size=window_size;self.padding=window_size//2;self.avg_pool=nn.AvgPool3d(self.window_size,1,self.padding)
    def forward(self,i1,i2):mu1=self.avg_pool(i1);mu2=self.avg_pool(i2);s1_sq=self.avg_pool(i1*i1)-mu1*mu1;s2_sq=self.avg_pool(i2*i2)-mu2*mu2;cov=self.avg_pool(i1*i2)-mu1*mu2;num=cov*cov;den=s1_sq*s2_sq;return 1-torch.mean(num/(den+1e-6))

def smooth_loss(displacement_field):
    dy=torch.abs(displacement_field[:,:,1:,:,:]-displacement_field[:,:,:-1,:,:]);dx=torch.abs(displacement_field[:,:,:,1:,:]-displacement_field[:,:,:,:-1,:]);dz=torch.abs(displacement_field[:,:,:,:,1:]-displacement_field[:,:,:,:,:-1]);return(torch.mean(dx**2)+torch.mean(dy**2)+torch.mean(dz**2))/3.0

# =================================================================================
# 2. DATASET-KLASSE mit korrigierter Dimensions-Logik
# =================================================================================
class PairwiseDataset(Dataset):
    def __init__(self, moving_zarr_path, fixed_zarr_path, fixed_index=0):
        self.moving_arr = zarr.open(moving_zarr_path, mode='r')
        self.fixed_arr = zarr.open(fixed_zarr_path, mode='r')
        self.num_images = self.moving_arr.shape[3]
        
        self.original_shape = self.moving_arr.shape # (H, W, D, T)
        self.padded_shape = [self.original_shape[2], self.original_shape[0], self.original_shape[1]] # D, H, W
        for i in range(3):
            if self.padded_shape[i] % 4 != 0:
                self.padded_shape[i] = (self.padded_shape[i] // 4 + 1) * 4
        
        self.fixed_np = self.fixed_arr[..., fixed_index]
        self.fixed_tensor = self._preprocess(self.fixed_np)

    def __len__(self):
        return self.num_images

    def __getitem__(self, idx):
        moving_np = self.moving_arr[..., idx]
        moving_tensor = self._preprocess(moving_np)
        return moving_tensor, self.fixed_tensor

    def _preprocess(self, volume_np):
        # Konvertiere (H, W, D) -> (1, D, H, W) für C,D,H,W Konvention
        tensor = torch.from_numpy(volume_np.astype(np.float32)).permute(2, 0, 1).unsqueeze(0)
        
        # Padding anwenden
        D_pad, H_pad, W_pad = self.padded_shape
        pad_d = D_pad - tensor.shape[1]
        pad_h = H_pad - tensor.shape[2]
        pad_w = W_pad - tensor.shape[3]
        padding = (pad_w // 2, pad_w - pad_w // 2, pad_h // 2, pad_h - pad_h // 2, pad_d // 2, pad_d - pad_d // 2)
        return F.pad(tensor, padding, "constant", 0)

# =================================================================================
# 3. HAUPT-TRAININGSSKRIPT mit korrigierter Tensor-Behandlung
# =================================================================================
if __name__ == '__main__':
    # --- Konfiguration ---
    MOVING_ZARR_PATH = "/media/shooty/Dev/repos/MRI-MoCoCo/MRI-Datasets/DCE"
    FIXED_ZARR_PATH = "/media/shooty/Dev/repos/MRI-MoCoCo/MRI-Datasets/mdreg_DCE_fitting_results/coreg_zarr_2.zarr"
    MODEL_SAVE_PATH = "./pairwise_model_final.pth"
    FIXED_IMAGE_INDEX = 50
    LEARNING_RATE = 1e-4
    NUM_EPOCHS = 50
    SMOOTH_REG_WEIGHT = 1e-3
    BATCH_SIZE = 2

    # --- Setup ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Verwende Gerät: {device}")

    dataset = PairwiseDataset(MOVING_ZARR_PATH, FIXED_ZARR_PATH, fixed_index=FIXED_IMAGE_INDEX)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

    # --- Modell initialisieren ---
    padded_input_size = tuple(dataset.padded_shape) # (D, H, W)
    
    model = UNet3D().to(device)
    stn = SpatialTransformer3D(size=padded_input_size).to(device)
    try:
        model = torch.compile(model)
        print("Modell optimiert.")
    except Exception as e:
        print(f"torch.compile() nicht verfügbar: {e}")
    
    similarity_loss_fn = LocalNCCLoss().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # --- Trainings-Schleife ---
    print(f"Starte Pairwise Training mit Batch-Größe: {BATCH_SIZE}")
    for epoch in range(NUM_EPOCHS):
        pbar = tqdm.tqdm(dataloader, desc=f"Epoche {epoch+1}/{NUM_EPOCHS}")
        for moving_batch, fixed_batch in pbar:
            # Daten vom DataLoader haben die Form (B, 1, D, H, W)
            # Wir bringen sie auf die GPU. Das `squeeze` ist nicht mehr nötig.
            moving_batch = moving_batch.to(device)
            fixed_batch = fixed_batch.to(device)

            optimizer.zero_grad()
            
            # Forward-Pass
            predicted_dvfs = model(fixed_batch, moving_batch)
            warped_images = stn(moving_batch, predicted_dvfs)
            
            # Verlustberechnung
            sim_loss = similarity_loss_fn(warped_images, fixed_batch)
            sm_loss = smooth_loss(predicted_dvfs)
            total_loss = sim_loss + SMOOTH_REG_WEIGHT * sm_loss
            
            total_loss.backward()
            optimizer.step()
            
            pbar.set_postfix(loss=total_loss.item())

    torch.save(model.state_dict(), MODEL_SAVE_PATH)
    print(f"✔️ Training abgeschlossen.")

Verwende Gerät: cuda
Modell optimiert.
Starte Pairwise Training mit Batch-Größe: 2


Epoche 50/50: 100%|██████████| 500/500 [01:53<00:00,  4.40it/s, loss=1]

✔️ Training abgeschlossen.


In [3]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import zarr
import tqdm
from torch.utils.data import Dataset, DataLoader

# =================================================================================
# 1. MODELL- & LOSS-DEFINITIONEN
# =================================================================================

class UNet3D(nn.Module):
    def __init__(self, in_channels=2, out_channels=3):
        super().__init__()
        self.enc1 = self._conv_block(in_channels, 16)
        self.pool1 = nn.MaxPool3d(2)
        self.enc2 = self._conv_block(16, 32)
        self.pool2 = nn.MaxPool3d(2)
        self.bottleneck = self._conv_block(32, 64)
        self.upconv2 = nn.ConvTranspose3d(64, 32, kernel_size=2, stride=2)
        self.dec2 = self._conv_block(64, 32)
        self.upconv1 = nn.ConvTranspose3d(32, 16, kernel_size=2, stride=2)
        self.dec1 = self._conv_block(32, 16)
        self.final_conv = nn.Conv3d(16, out_channels, kernel_size=1)
        self.final_conv.weight.data.zero_()
        self.final_conv.bias.data.zero_()

    def _conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv3d(in_c, out_c, 3, 1, 1, bias=False), nn.InstanceNorm3d(out_c), nn.ReLU(True),
            nn.Conv3d(out_c, out_c, 3, 1, 1, bias=False), nn.InstanceNorm3d(out_c), nn.ReLU(True)
        )

    def forward(self, x_fixed, x_moving):
        x = torch.cat([x_fixed, x_moving], dim=1)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))
        d2 = self.upconv2(b)
        if d2.shape[2:] != e2.shape[2:]:
            d2 = F.interpolate(d2, size=e2.shape[2:], mode='trilinear', align_corners=False)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.upconv1(d2)
        if d1.shape[2:] != e1.shape[2:]:
            d1 = F.interpolate(d1, size=e1.shape[2:], mode='trilinear', align_corners=False)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.final_conv(d1)

class SpatialTransformer3D(nn.Module):
    def __init__(self, size):
        super().__init__()
        vectors = [torch.arange(0, s) for s in size]
        grid = torch.stack(torch.meshgrid(vectors, indexing='ij'), 0)
        self.register_buffer('grid', grid.unsqueeze(0).float(), persistent=False)

    def forward(self, src, flow):
        new_locs = self.grid + flow
        shape = flow.shape[2:]
        for i in range(len(shape)):
            new_locs[:, i, ...] = 2 * (new_locs[:, i, ...] / (shape[i] - 1) - 0.5)
        return F.grid_sample(src, new_locs.permute(0, 2, 3, 4, 1)[..., [2, 1, 0]], align_corners=True, padding_mode="border")

class LocalNCCLoss(nn.Module):
    def __init__(self, window_size=9):
        super().__init__()
        self.avg_pool = nn.AvgPool3d(window_size, 1, window_size // 2)
    def forward(self, i1, i2):
        mu1, mu2 = self.avg_pool(i1), self.avg_pool(i2)
        i1_sq, i2_sq = self.avg_pool(i1**2), self.avg_pool(i2**2)
        i12 = self.avg_pool(i1 * i2)
        s1_sq, s2_sq = i1_sq - mu1**2, i2_sq - mu2**2
        cov = i12 - mu1 * mu2
        return 1 - torch.mean((cov**2) / (s1_sq * s2_sq + 1e-6))

def smooth_loss(disp):
    """ Berechnet den Glättungsverlust als den mittleren quadratischen Gradienten des Verschiebungsfeldes. """
    # disp shape ist (B, C, D, H, W)
    # Die Gradienten werden entlang der räumlichen Dimensionen D, H, W (Indizes 2, 3, 4) berechnet.
    
    # Gradient entlang der Tiefe (Achse 2)
    dz = torch.abs(disp[:, :, 1:, :, :] - disp[:, :, :-1, :, :])
    
    # Gradient entlang der Höhe (Achse 3)
    dy = torch.abs(disp[:, :, :, 1:, :] - disp[:, :, :, :-1, :])
    
    # Gradient entlang der Breite (Achse 4)
    dx = torch.abs(disp[:, :, :, :, 1:] - disp[:, :, :, :, :-1])
    
    # Berechne den mittleren quadratischen Gradienten und normalisiere
    return (torch.mean(dx**2) + torch.mean(dy**2) + torch.mean(dz**2)) / 3.0

# =================================================================================
# 2. DATASET-KLASSE FÜR GROUP-WISE
# =================================================================================
class GroupwiseDataset(Dataset):
    def __init__(self, zarr_path):
        self.zarr_array = zarr.open(zarr_path, mode='r')
        self.padded_shape = self._get_padded_shape()

    def __len__(self):
        return 1 # Wir laden immer die gesamte Gruppe

    def __getitem__(self, idx):
        full_series_np = self.zarr_array[:] # Form (H, W, D, T)
        series_tensor = torch.from_numpy(full_series_np.astype(np.float32)).permute(3, 2, 0, 1) # -> (T, D, H, W)
        series_tensor = series_tensor.unsqueeze(1) # -> (T, C, D, H, W)
        
        # Padding
        D_orig, H_orig, W_orig = series_tensor.shape[2:]
        D_pad, H_pad, W_pad = self.padded_shape
        pad_dims = (W_pad//2 - W_orig//2, W_pad - W_orig - (W_pad//2 - W_orig//2),
                    H_pad//2 - H_orig//2, H_pad - H_orig - (H_pad//2 - H_orig//2),
                    D_pad//2 - D_orig//2, D_pad - D_orig - (D_pad//2 - D_orig//2))
        return F.pad(series_tensor, pad_dims, "constant", 0)
        
    def _get_padded_shape(self):
        shape = self.zarr_array.shape # (H, W, D, T)
        padded_shape = [shape[2], shape[0], shape[1]] # D, H, W
        for i in range(3):
            if padded_shape[i] % 4 != 0:
                padded_shape[i] = (padded_shape[i] // 4 + 1) * 4
        return tuple(padded_shape)

# =================================================================================
# 3. HAUPT-TRAININGSSKRIPT (GROUP-WISE)
# =================================================================================
# =================================================================================
# 3. FINALES TRAININGSSKRIPT
# =================================================================================
if __name__ == '__main__':
    # --- Konfiguration ---
    ZARR_PATH = "/media/shooty/Dev/repos/MRI-MoCoCo/MRI-Datasets/DCE"
    MODEL_SAVE_PATH = "./groupwise_model_final.pth"
    LEARNING_RATE = 1e-4
    NUM_EPOCHS = 50
    SMOOTH_REG_WEIGHT = 1e-4 # Starten Sie mit einem kleinen Wert
    CYCLIC_REG_WEIGHT = 1e-2 # Oft höher als der Smooth-Wert
    MINI_BATCH_SIZE = 2      # Beginnen Sie mit 1 oder 2, um Speicherprobleme auszuschließen

    # --- Setup ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Verwende Gerät: {device}")

    dataset = GroupwiseDataset(ZARR_PATH)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

    # --- Modell initialisieren ---
    padded_input_size = dataset.padded_shape # (D, H, W)
    model = UNet3D().to(device)
    stn = SpatialTransformer3D(size=padded_input_size).to(device)
    
    try:
        model = torch.compile(model)
        print("U-Net Modell erfolgreich optimiert.")
    except Exception as e:
        print(f"torch.compile() nicht verfügbar oder fehlgeschlagen: {e}")
    
    similarity_loss_fn = LocalNCCLoss().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # --- Trainings-Schleife ---
    print(f"Starte Group-wise Training mit Mini-Batch-Größe: {MINI_BATCH_SIZE}")
    for epoch in range(NUM_EPOCHS):
        
        # Lade die gesamte Bildgruppe einmal pro Epoche auf die CPU
        image_group_cpu = next(iter(dataloader)).squeeze(0) # Form (T, C, D_pad, H_pad, W_pad)
        
        # Berechne das Template auf der CPU
        with torch.no_grad():
            template_cpu = torch.mean(image_group_cpu, dim=0, keepdim=True)
        
        # Setze den Optimizer zurück
        optimizer.zero_grad()
        
        # Initialisiere die Akkumulatoren für die Epoche
        # Wichtig: Diese sind jetzt normale Python-Floats für die Anzeige
        epoch_sim_loss = 0.0
        epoch_smooth_loss = 0.0
        
        # Dieser Tensor sammelt ALLE Verlustkomponenten und behält den Graphen
        total_epoch_loss = torch.tensor(0.0, device=device)
        
        num_images_in_group = image_group_cpu.shape[0]
        num_accumulation_steps = -(-num_images_in_group // MINI_BATCH_SIZE) # Saubere Division nach oben
        pbar = tqdm.tqdm(range(0, num_images_in_group, MINI_BATCH_SIZE), desc=f"Epoche {epoch+1}/{NUM_EPOCHS}")
        
        # Mini-Batch-Schleife
        for i in pbar:
            # Nur das Mini-Batch auf die GPU laden
            sub_batch_images = image_group_cpu[i : i + MINI_BATCH_SIZE].to(device)
            sub_template_batch = template_cpu.repeat(sub_batch_images.shape[0], 1, 1, 1, 1).to(device)

            # Forward-Pass
            predicted_dvfs = model(sub_template_batch, sub_batch_images)
            warped_images = stn(sub_batch_images, predicted_dvfs)

            # Berechne die einzelnen Verlust-Komponenten für diesen Mini-Batch
            sim_loss = similarity_loss_fn(warped_images, sub_template_batch)
            sm_loss = smooth_loss(predicted_dvfs)
            
            # WICHTIG: Wir berechnen den zyklischen Verlust FÜR JEDEN MINI-BATCH
            # und summieren ihn auf. Die Summe der DVFs wird nicht mehr explizit gespeichert.
            # Dies hält den Graphen für alle Teile des Verlusts intakt.
            # Der zyklische Loss bestraft die Netto-Verschiebung im Batch.
            cyclic_loss = torch.mean(torch.sum(predicted_dvfs, dim=0)**2)
            
            # Addiere alle gewichteten Verluste dieses Mini-Batches zum Gesamtverlust der Epoche
            # (Skalierung mit 1/num_steps ist äquivalent zur Mittelwertbildung am Ende)
            total_minibatch_loss = (sim_loss + SMOOTH_REG_WEIGHT * sm_loss + CYCLIC_REG_WEIGHT * cyclic_loss) / num_accumulation_steps
            
            # Füge den Mini-Batch-Verlust zum Gesamtverlust der Epoche hinzu
            total_epoch_loss += total_minibatch_loss
            
            # Sammle skalare Werte nur für die Anzeige
            epoch_sim_loss += sim_loss.item()
            epoch_smooth_loss += sm_loss.item()
            pbar.set_postfix(sim_loss=sim_loss.item(), sm_loss=sm_loss.item(), cyclic_loss=cyclic_loss.item())

        # ===================================================================
        # EIN EINZIGER BACKWARD-PASS PRO EPOCHE
        # ===================================================================
        # Rufe .backward() auf dem finalen, akkumulierten Gesamtverlust auf
        total_epoch_loss.backward()
        
        # Aktualisiere die Gewichte einmal
        optimizer.step()
        
        # Gib die durchschnittlichen Verluste für die Epoche aus
        avg_sim = epoch_sim_loss / num_accumulation_steps
        avg_smooth = epoch_smooth_loss / num_accumulation_steps
        # Der letzte berechnete cyclic_loss dient als Indikator
        print(f"Epoche {epoch+1} abgeschlossen - Avg Sim Loss: {avg_sim:.4f}, Avg Smooth Loss: {avg_smooth:.4f}, Last Cyclic Loss: {cyclic_loss.item():.4f}")

    torch.save(model.state_dict(), MODEL_SAVE_PATH)
    print(f"✔️ Training abgeschlossen.")

Verwende Gerät: cuda
U-Net Modell erfolgreich optimiert.
Starte Group-wise Training mit Mini-Batch-Größe: 2


Epoche 1/50:   0%|          | 2/500 [00:00<01:57,  4.24it/s, cyclic_loss=0, sim_loss=0.315, sm_loss=0]


OutOfMemoryError: CUDA out of memory. Tried to allocate 416.00 MiB. GPU 0 has a total capacity of 23.53 GiB of which 345.88 MiB is free. Process 200442 has 23.14 GiB memory in use. Of the allocated memory 22.59 GiB is allocated by PyTorch, and 104.99 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)